In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = '/content/yellow_tripdata_2024-03.parquet'
urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

print(pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine))

   n_rows
0  486964


In [3]:
from sqlalchemy import text

sql_setup = """
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
"""

# Виконуємо DDL-команди (створення таблиць)
with engine.begin() as conn:
    conn.execute(text(sql_setup))

# Виконуємо SELECT і виводимо як DataFrame
sql_query = """
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;
"""

pd.read_sql(sql_query, engine)

,pickup_zone_id,n_days
0,1,5
1,3,6
2,4,6
3,6,4
4,7,6
5,8,1
6,9,5
7,10,6
8,11,4
9,12,5


In [4]:
sql_setup_typed = """
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;
"""

# Виконуємо DDL-команди
with engine.begin() as conn:
    conn.execute(text(sql_setup_typed))

# Перевіряємо типи через SELECT
sql_query_typed = """
SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5;
"""

pd.read_sql(sql_query_typed, engine)

,pickup_at,pickup_type
0,2024-03-01 05:18:51+00:00,timestamp with time zone
1,2024-03-01 05:26:00+00:00,timestamp with time zone
2,2024-03-01 05:09:22+00:00,timestamp with time zone
3,2024-03-01 05:33:45+00:00,timestamp with time zone
4,2024-03-01 05:05:43+00:00,timestamp with time zone


In [5]:
sql_task3_setup = """
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
"""

with engine.begin() as conn:
    conn.execute(text(sql_task3_setup))

pd.read_sql("""
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
""", engine)

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [6]:
# Query 1. Containment @>
print("--- Query 1: Containment ---")
display(pd.read_sql("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 5;
""", engine))

# Query 2. Key existence
print("\n--- Query 2: Key existence ---")
display(pd.read_sql("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 5;
""", engine))

# Query 3. JSONPath
print("\n--- Query 3: JSONPath ---")
display(pd.read_sql("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 5;
""", engine))

--- Query 1: Containment ---


,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."



--- Query 2: Key existence ---


,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."



--- Query 3: JSONPath ---


,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11


In [7]:
def print_explain(sql_query):
    with engine.connect() as conn:
        res = conn.execute(text(sql_query))
        print('\n'.join(row[0] for row in res))
        print("-" * 60)

# Очищення старих індексів
with engine.begin() as conn:
    conn.execute(text("""
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
        DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
        DROP INDEX IF EXISTS idx_t7_events_device_btree;
        ANALYZE t7_hw_events;
    """))

print("1. Containment query БЕЗ GIN-індексу (очікуємо Seq Scan):")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

# Створення GIN-індексу
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_payload_gin_pathops
        ON t7_hw_events USING GIN (payload jsonb_path_ops);
        ANALYZE t7_hw_events;
    """))

print("2. Containment query ПІСЛЯ створення GIN-індексу (очікуємо Bitmap Index Scan):")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

print("3. Path-equality query БЕЗ Expression B-tree індексу (очікуємо Seq Scan, бо GIN не підходить):")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

# Створення Expression B-tree індексу
with engine.begin() as conn:
    conn.execute(text("""
        CREATE INDEX idx_t7_events_device_btree
        ON t7_hw_events ((payload ->> 'device'));
        ANALYZE t7_hw_events;
    """))

print("4. Path-equality query ПІСЛЯ створення Expression B-tree (очікуємо Index Scan / Bitmap Index Scan):")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

1. Containment query БЕЗ GIN-індексу (очікуємо Seq Scan):
Aggregate  (cost=1759.01..1759.02 rows=1 width=8) (actual time=105.629..105.630 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3605 width=0) (actual time=0.022..105.108 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.259 ms
Execution Time: 105.662 ms
------------------------------------------------------------
2. Containment query ПІСЛЯ створення GIN-індексу (очікуємо Bitmap Index Scan):
Aggregate  (cost=1106.74..1106.75 rows=1 width=8) (actual time=31.147..31.149 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=37.67..1096.88 rows=3947 width=0) (actual time=4.838..30.798 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
    

In [8]:
# Крок 1. Створення нормалізованої таблиці
sql_task5_step1 = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);
ANALYZE t7_hw_checkouts_norm;
"""

with engine.begin() as conn:
    conn.execute(text(sql_task5_step1))

print("Кількість нормалізованих записів:")
display(pd.read_sql("SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;", engine))

# Крок 2. Порівняння EXPLAIN ANALYZE
print("\n--- JSONB-версія ---")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
""")

print("\n--- Normalized-версія ---")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
""")

Кількість нормалізованих записів:


,n_checkouts
0,10000



--- JSONB-версія ---
Limit  (cost=1957.03..1957.04 rows=3 width=72) (actual time=50.910..50.915 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1957.03..1970.99 rows=5584 width=72) (actual time=50.908..50.911 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1801.10..1884.86 rows=5584 width=72) (actual time=50.840..50.888 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.16 rows=10063 width=132) (actual time=0.020..39.392 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 0.269 ms

# Крок 1. Створення нормалізованої таблиці
sql_task5_step1 = """
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);
ANALYZE t7_hw_checkouts_norm;
"""

with engine.begin() as conn:
    conn.execute(text(sql_task5_step1))

print("Кількість нормалізованих записів:")
display(pd.read_sql("SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;", engine))

# Крок 2. Порівняння EXPLAIN ANALYZE
print("\n--- JSONB-версія ---")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
""")

print("\n--- Normalized-версія ---")
print_explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
""")

# Порівняння підходів:

*   PostgreSQL JSONB: Зручний, коли потрібні SQL, JOIN, транзакції та зв’язок із нормалізованими таблицями.

*   MongoDB aggregation pipeline: Природний для document-first workloads.
*   Сирий масивний clickstream краще тримати в document store (MongoDB) або data lake.


*   Підготовлені ML-ознаки (curated ML features) зручніше тримати у типізованих SQL-таблицях.


---





# Reflection

1.   Аналіз плану виконання підтверджує, що до створення GIN-індексу запит на перевірку payload @> ... виконувався через Seq Scan, послідовно зчитуючи всі 50 000 рядків. Після додавання індексу оптимізатор очікувано перейшов на Bitmap Index Scan та Bitmap Heap Scan. Це суттєво зменшило показник cost (з 1759.02 до 1106.75) і фактичний час виконання (зі 105.6 ms до 31.1 ms), оскільки база тепер точково звертається лише до потрібних сторінок пам'яті.

1.   GIN-індекс (jsonb_path_ops або jsonb_ops) оптимізований для перевірки структури JSON, наприклад, наявності ключів чи входження цілих об'єктів (@>). Запит payload ->> 'device' = 'mobile' працює інакше: він дістає значення за вказаним шляхом, перетворює його на тип text і виконує стандартне рядкове порівняння. GIN не розуміє таких текстових операцій, тому для швидкого пошуку за цим патерном створюється класичний Expression B-tree індекс на сам вираз.

2.   Головний trade-off GIN-індексу: він кардинально пришвидшує читання напівструктурованих даних, але робить операції запису (INSERT, UPDATE) значно дорожчими. Базі даних доводиться розбирати кожен JSON-документ і оновлювати інвертований індекс для множини ключів та значень. Крім того, такий індекс займає чимало місця на диску порівняно зі звичайним B-tree.


2.   У ML-інфраструктурі JSONB ідеально підходить для "гнучких" даних: логів сирих подій (clickstream), метаданих реєстратури моделей (Model Registry) або гіперпараметрів, де схема часто змінюється. Нормалізовані таблиці є безальтернативним вибором для Feature Store — підготовлених ML-ознак, де потрібні сувора типізація, гарантія якості даних та блискавичні JOIN для inference.

1.   Щоб додати подію refund у JSONB-схемі, жодні міграції не потрібні: застосунок просто починає писати нові JSON-об'єкти з event_type = 'refund'. У нормалізованому підході знадобиться DDL-міграція: доведеться робити ALTER TABLE для зміни існуючих CHECK-обмежень і додавання нових колонок (наприклад, refund_reason), або ж створювати окрему таблицю спеціально під повернення коштів.



